# Actor extraction accuracy: Mistral-7B

## Setup

In [ ]:
# packages

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import os
import csv
import json
import re

In [ ]:
os.getcwd()

In [ ]:
# read excel file
df = pd.read_excel('actor_names_researcher_mistral_manual.xlsx')
# make article_id int
df['article_id'] = df['article_id'].astype(int)

In [ ]:
df = df[df['double_coding'] == 0]

In [ ]:
df.actor_name = df.actor_name.str.strip()
df.actor_function = df.actor_function.str.strip()
df.actor_name_Mistral = df.actor_name_Mistral.str.strip()
df.actor_function_mistral = df.actor_function_mistral.str.strip()

In [ ]:
# create a column actor_coded, where actor_name is null then 0 else 1
df['actor_coded'] = np.where(df['actor_name'].isnull(), 0, 1)
df['actor_coded_mistral'] = np.where(df['actor_name_Mistral'].isnull(), 0, 1)

In [ ]:
# crosstab actor_coded and actor_coded_mistral
pd.crosstab(df.actor_coded, df.actor_coded_mistral)

In [ ]:
221/(221+80)

In [ ]:
not_coded = df[(df['actor_coded'] == 1) & (df['actor_coded_mistral'] == 0)]

In [ ]:
not_coded_functioncounts = not_coded.actor_function.value_counts(dropna=False).reset_index()
not_coded_functioncounts.columns = ['actor_function', 'count']

In [ ]:
coded = df[(df['actor_coded'] == 1)]
print(coded.shape)

coded_function_counts = coded.actor_function.value_counts(dropna=False).reset_index()
coded_function_counts.columns = ['actor_function', 'count']

In [ ]:
merged_function_counts = pd.merge(coded_function_counts, not_coded_functioncounts, on='actor_function', how='outer', suffixes=('_coded', '_not_coded')).fillna(0) 
merged_function_counts['percentage'] = merged_function_counts['count_not_coded'] / merged_function_counts['count_coded'] * 100
merged_function_counts.sort_values(by='percentage', ascending=False)

merged_function_counts.to_excel('notcoded_actor_functions_mistral.xlsx', index=False)

In [ ]:
# calculate the accuracy
accuracy = (221) / (221+80+63)
print(accuracy)

In [ ]:
df_persons = df[df['actor_type'] == 'Persoon']
df_organizations = df[df['actor_type'] == 'Organisatie']

In [ ]:
pd.crosstab(df_persons.actor_coded, df_persons.actor_coded_mistral)

In [ ]:
153/(153+46)

In [ ]:
pd.crosstab(df_organizations.actor_coded, df_organizations.actor_coded_mistral)

In [ ]:
68/(68+34)

In [ ]:
manually_coded = df[df['actor_coded'] == 1]

In [ ]:
not_coded = df[(df['actor_coded'] == 1) & (df['actor_coded_mistral'] == 0)]

In [ ]:
not_coded = not_coded[['article_id', 'actor_name', 'actor_type']].drop_duplicates()
# save excel
not_coded.to_excel('actor_names_not_coded_mistral.xlsx', index=False)

In [ ]:
not_coded.actor_name.value_counts(dropna=False).head(50)

In [ ]:
extra_coded = df[(df['actor_coded'] == 0) & (df['actor_coded_mistral'] == 1)]

In [ ]:
extra_coded = extra_coded[['article_id', 'actor_name_Mistral']].drop_duplicates()
# save excel
extra_coded.to_excel('actor_names_extra_coded_mistral.xlsx', index=False)

In [ ]:
extra_coded[extra_coded['actor_name_Mistral'].str.contains('RIVM', na=False)]

In [ ]:
print(extra_coded.shape)

In [ ]:
46/(46+34)

In [ ]:
manually_coded_functions = manually_coded.actor_function.value_counts(dropna=False).reset_index().rename(columns={'index': 'actor_function', 'actor_function': 'count'})
manually_coded_functions.to_excel('manually_coded_functions.xlsx', index=False)

In [ ]:
not_coded_mistral = not_coded.actor_function.value_counts(dropna=False).reset_index().rename(columns={'index': 'actor_function_mistral', 'actor_function_mistral': 'count'})
not_coded_mistral.to_excel('not_coded_mistral_functions.xlsx', index=False)

In [ ]:
coded_by_both = df[(df['actor_coded'] == 1) & (df['actor_coded_mistral'] == 1)]

In [ ]:
coded_by_both['actor_function_new'] = coded_by_both['actor_function'].str.strip().copy()

In [ ]:
# change actor function categories
coded_by_both.loc[coded_by_both.actor_function_new == 'NL - Nationale regering - executive / uitvoerende macht', 'actor_function_new'] = 'A'
coded_by_both.loc[coded_by_both.actor_function_new == 'NL - Nationaal parlement en nationale partijen ‚Äì wetgevende macht', 'actor_function_new'] = 'A'
coded_by_both.loc[coded_by_both.actor_function_new == 'NL - Nationale regionale en lokale politieke organisaties en hun ambtenaren', 'actor_function_new'] = 'A'
coded_by_both.loc[coded_by_both.actor_function_new == 'NL - Nationale staatsorganisaties en hun ambtenaren', 'actor_function_new'] = 'A'
coded_by_both.loc[coded_by_both.actor_function_new == 'NL - Nationale koninklijke familie en haar leden', 'actor_function_new'] = 'A'
coded_by_both.loc[coded_by_both.actor_function_new == 'Regeringen/regeringsleiders/regeringsleden en/of andere politici in een ander land dan NL, op nationaal of lokaal niveau OF staatsorganisaties en hun ambtenaren', 'actor_function_new'] = 'A'
coded_by_both.loc[coded_by_both.actor_function_new == 'EU-instellingen en Internationale overheidsorganisaties (ook IGO‚Äôs) en hun leden', 'actor_function_new'] = 'A'
coded_by_both.loc[coded_by_both.actor_function_new == 'Nationale en internationale rechterlijke macht', 'actor_function_new'] = 'A'
coded_by_both.loc[coded_by_both.actor_function_new == 'Wetenschappelijke/medische organisaties en onderzoekers', 'actor_function_new'] = 'B'
coded_by_both.loc[coded_by_both.actor_function_new == 'Openbare en semiopenbare instellingen', 'actor_function_new'] = 'B'
coded_by_both.loc[coded_by_both.actor_function_new == 'Zakelijke organisaties en hun werknemers', 'actor_function_new'] = 'B'
coded_by_both.loc[coded_by_both.actor_function_new == 'Bekende mediapersonen (anders dan journalisten)', 'actor_function_new'] = 'B'
coded_by_both.loc[coded_by_both.actor_function_new == 'Journalisten anders dan de schrijver van het huidige artikel of nieuwsorganisaties anders dan de nieuwsorganisatie van het huidige artikel.', 'actor_function_new'] = 'B'
coded_by_both.loc[coded_by_both.actor_function_new == 'Niet-governementele organisaties (NGO), maatschappelijke organisaties, en hun leden', 'actor_function_new'] = 'C'
coded_by_both.loc[coded_by_both.actor_function_new == 'Religieuze instellingen en hun leden (ook gelovigen)', 'actor_function_new'] = 'C'
coded_by_both.loc[coded_by_both.actor_function_new == 'Publiek en leden van het publiek, publieke opiniepeilingen en hun respondenten', 'actor_function_new'] = 'D'

# see where actor_function_new is NaN
coded_by_both[coded_by_both['actor_function_new'].isna() == True]

In [ ]:
pd.crosstab(coded_by_both.actor_function_new, coded_by_both.actor_function)

In [ ]:
# drop if actor_function_mistral is NaN
coded_by_both = coded_by_both.dropna(subset=['actor_function_mistral'])

In [ ]:
# classification report
print(classification_report(coded_by_both.actor_function_new, coded_by_both.actor_function_mistral))

In [ ]:
# crosstab actor_function_new and actor_function_mistral
pd.crosstab(coded_by_both.actor_function_new, coded_by_both.actor_function_mistral)

In [ ]:
# see where coded_by_both is 3 and actor_function_new is not 3
coded_by_both[(coded_by_both['actor_function_mistral'] != 'D') & (coded_by_both['actor_function_new'] == 'D')]

In [ ]:
coded_by_both[(coded_by_both['actor_function_mistral'] == 'D') & (coded_by_both['actor_function_new'] == 'D')]

In [ ]:
coded_by_both[(coded_by_both['actor_function_mistral'] != 'C') & (coded_by_both['actor_function_new'] == 'C')]

In [ ]:
# combine C and D in both actor_function_new and actor_function_mistral
actors_coded_both.loc[actors_coded_both.actor_function_new == 'D', 'actor_function_new'] = 'C'
actors_coded_both.loc[actors_coded_both.actor_function_mistral == 'D', 'actor_function_mistral'] = 'C'

# classification report
print(classification_report(actors_coded_both.actor_function_new, actors_coded_both.actor_function_mistral))